Copyright (c) Meta Platforms, Inc. and affiliates.

<a target="_blank" href="https://colab.research.google.com/github/facebookresearch/co-tracker/blob/main/notebooks/demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# CoTracker: It is Better to Track Together
This is a demo for <a href="https://co-tracker.github.io/">CoTracker</a>, a model that can track any point in a video.

<img src="https://www.robots.ox.ac.uk/~nikita/storage/cotracker/bmx-bumps.gif" alt="Logo" width="50%">

Don't forget to turn on GPU support if you're running this demo in Colab.

**Runtime** -> **Change runtime type** -> **Hardware accelerator** -> **GPU**

Let's install dependencies for Colab:

In [11]:
# !git clone https://github.com/facebookresearch/co-tracker
# %cd co-tracker
# !pip install -e .
# !pip install opencv-python matplotlib moviepy flow_vis
# !mkdir checkpoints
# %cd checkpoints
# !wget https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth

In [12]:
%cd ..
import os
import torch

from base64 import b64encode
from cotracker.utils.visualizer import Visualizer, read_video_from_path
from IPython.display import HTML, Video, display


/


Read a video from CO3D:

In [13]:
video = read_video_from_path('/content/occlusion_9sec.mp4')
video = torch.from_numpy(video).permute(0, 3, 1, 2)[None].float()

Import CoTrackerPredictor and create an instance of it. We'll use this object to estimate tracks:

In [14]:
from cotracker.predictor import CoTrackerPredictor

model = torch.hub.load("facebookresearch/co-tracker", "cotracker3_online")

Using cache found in /root/.cache/torch/hub/facebookresearch_co-tracker_main


In [15]:
if torch.cuda.is_available():
    model = model.cuda()
    video = video.cuda()

## Regular grid + Segmentation mask

Let's now sample points on a grid and filter them with a segmentation mask.
This allows us to track points sampled densely on an object because we consume less GPU memory.

In [16]:
import numpy as np
from PIL import Image
import torch.nn.functional as F

from cotracker.predictor import CoTrackerOnlinePredictor
from cotracker.models.core.model_utils import get_points_on_a_grid

grid_size = 100
grid_query_frame = 0  # query time for the regular grid (same as offline `grid_query_frame`)
online_model = torch.hub.load("facebookresearch/co-tracker", "cotracker3_online")
if torch.cuda.is_available():
    online_model = online_model.cuda()


Using cache found in /root/.cache/torch/hub/facebookresearch_co-tracker_main


In [17]:
input_mask = '/content/mask.png'
segm_mask = np.array(Image.open(input_mask))

That's a segmentation mask for the first frame:

In [18]:
# plt.imshow((segm_mask[...,None]/255.*video[0,0].permute(1,2,0).cpu().numpy()/255.))

In [19]:
# Online model: same mask logic as offline CoTrackerPredictor._compute_sparse_tracks,
# then express queries in original (H, W) pixels — CoTrackerOnlinePredictor scales them on is_first_step.
device = video.device
B, T_len, C, H, W = video.shape
interp_h, interp_w = online_model.interp_shape

segm_mask_t = torch.from_numpy(segm_mask.astype(np.float32))
if segm_mask_t.ndim == 2:
    segm_mask_t = segm_mask_t[None, None]
elif segm_mask_t.ndim == 3:
    segm_mask_t = segm_mask_t[..., 0:1].permute(2, 0, 1).unsqueeze(0)
else:
    segm_mask_t = segm_mask_t[None, None]

grid_pts = get_points_on_a_grid(grid_size, (interp_h, interp_w), device=device)
segm_interp = F.interpolate(
    segm_mask_t.to(device), size=(interp_h, interp_w), mode="nearest"
)
yi = grid_pts[0, :, 1].round().long().clamp(0, interp_h - 1)
xi = grid_pts[0, :, 0].round().long().clamp(0, interp_w - 1)
point_mask = segm_interp[0, 0][yi, xi].bool()
grid_pts_f = grid_pts[:, point_mask]
if grid_pts_f.shape[1] == 0:
    raise ValueError(
        "No grid points under the mask; try a larger grid_size or check mask/video alignment."
    )

sx = float(W - 1) / float(interp_w - 1) if interp_w > 1 else 1.0
sy = float(H - 1) / float(interp_h - 1) if interp_h > 1 else 1.0
xy_orig = grid_pts_f.clone()
xy_orig[..., 0] = grid_pts_f[..., 0] * sx
xy_orig[..., 1] = grid_pts_f[..., 1] * sy

queries = torch.cat(
    [
        torch.full_like(xy_orig[..., :1], float(grid_query_frame)),
        xy_orig,
    ],
    dim=-1,
).repeat(B, 1, 1)

step = online_model.step
window_frames = []


def _process_step_online(window_frames, is_first_step):
    win = window_frames[-step * 2 :]
    video_chunk = (
        torch.tensor(np.stack(win), device=device).float().permute(0, 3, 1, 2)[None]
    )
    return online_model(
        video_chunk,
        is_first_step=is_first_step,
        queries=queries,
        add_support_grid=True,
    )


is_first_step = True
i_last = 0
for i_last in range(T_len):
    frame = video[0, i_last].permute(1, 2, 0).detach().cpu().numpy().astype(np.float32)
    if i_last % step == 0 and i_last != 0:
        pred_tracks, pred_visibility = _process_step_online(window_frames, is_first_step)
        is_first_step = False
    window_frames.append(frame)

pred_tracks, pred_visibility = _process_step_online(
    window_frames[-(i_last % step) - step - 1 :],
    is_first_step,
)

vis = Visualizer(
    save_dir="./videos",
    pad_value=100,
    linewidth=2,
)
vis.visualize(
    video=video,
    tracks=pred_tracks,
    visibility=pred_visibility,
    query_frame=grid_query_frame,
    filename="segm_grid",
)


Video saved to ./videos/segm_grid.mp4


tensor([[[[[255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           ...,
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255]],

          [[255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           ...,
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255]],

          [[255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           ...,
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255]]],


         [[[255, 255, 255,  ..., 255, 255, 255],
           [2

We are now only tracking points on the object (and around):

In [20]:
display(Video("./videos/segm_grid.mp4", embed=True))


NameError: name 'Video' is not defined